### Use Google Colab, with GPU

In [12]:
"Uncomment below to mount the drive to google Drive, remember to upload the pictures to your google Drive for this code to work, below you can see the path Rasmus used"
"Poista kommentti merkki yhdistääksesi google Driveen kuvia varten, muista ladata kuvat myös Driveen oikeaan kansioon, alla näkyy Rasmuksen käyttämä polku"
#from google.colab import drive
#drive.mount('/content/drive')

'Poista kommentti merkki yhdistääksesi google Driveen kuvia varten, muista ladata kuvat myös Driveen oikeaan kansioon, alla näkyy Rasmuksen käyttämä polku'

In [13]:
# cell 1: imports & basic config
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
import wandb
import gradio as gr

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# paths
root = Path('/content/drive/MyDrive/2025/Trees')  # notebook is in TREES/
images_dir = root / "images"

print(root)
print(images_dir)

print(f"PyTorch version: {torch.__version__}")
print(f"Gradio version: {gr.__version__}")

Device: cpu
/content/drive/MyDrive/2025/Trees
/content/drive/MyDrive/2025/Trees/images
PyTorch version: 2.5.1
Gradio version: 5.47.0


In [14]:
!wandb login

wandb: Currently logged in as: iiro23001 (iiro23001-hamk) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [15]:
# Initialize W&B and set up the project
wandb.init(project="ResNet18_Project", config={
    "epochs": 15,
    "batch_size": 16,
    "learning_rate": 0.001,
    "architecture": "ResNet18",
    "dataset": "RPS",
    "image_size": 224
})

# Get the config from W&B for easy access
config = wandb.config

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [17]:
# CORRECT transformations for a pretrained ResNet model
transform = Compose([
    Resize((config.image_size, config.image_size)), # Use 224x224
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406],         # Standard ImageNet mean
              std=[0.229, 0.224, 0.225])          # Standard ImageNet std dev
])

# Load the datasets

val_data = datasets.ImageFolder(images_dir, transform=transform)

# Create data loaders
train_loader = DataLoader(train_data, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=config.batch_size, shuffle=False)

# Print dataset information
print(f"Classes: {train_data.classes}")
print(f"Number of training samples: {len(train_data)}")
print(f"Number of validation samples: {len(val_data)}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/2025/Trees/images'

In [ ]:
# Load a pretrained ResNet18 model
# model = models.resnet50(pretrained=True)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Modify the final fully connected layer for our 3 classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(train_data.classes))

# Move the model to the configured device
model = model.to(device)

print("ResNet18 model loaded and modified for 5 classes.")

ResNet18 model loaded and modified for 5 classes.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

In [ ]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_accuracy = 100. * correct / total
    return epoch_loss, epoch_accuracy

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_accuracy = 100. * correct / total
    return epoch_loss, epoch_accuracy

In [ ]:
# Log gradients and model parameters to W&B
wandb.watch(model, criterion, log="all", log_freq=10)

print("Starting model training...")
for epoch in range(config.epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    # Log metrics to W&B
    wandb.log({
        "Epoch": epoch + 1,
        "Train Loss": train_loss,
        "Train Accuracy": train_acc,
        "Validation Loss": val_loss,
        "Validation Accuracy": val_acc
    })

    print(f'Epoch {epoch+1}/{config.epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')

print("Training finished.")

Starting model training...
Epoch 1/15 | Train Loss: 0.4709 | Train Acc: 84.56% | Val Loss: 3.6229 | Val Acc: 38.97%
Epoch 2/15 | Train Loss: 0.3901 | Train Acc: 90.44% | Val Loss: 22.2234 | Val Acc: 34.56%
Epoch 3/15 | Train Loss: 0.1157 | Train Acc: 96.32% | Val Loss: 0.8456 | Val Acc: 76.47%
Epoch 4/15 | Train Loss: 0.1073 | Train Acc: 96.32% | Val Loss: 0.8755 | Val Acc: 79.41%
Epoch 5/15 | Train Loss: 0.1792 | Train Acc: 96.32% | Val Loss: 0.4677 | Val Acc: 87.50%
Epoch 6/15 | Train Loss: 0.1322 | Train Acc: 95.59% | Val Loss: 0.0322 | Val Acc: 98.53%
Epoch 7/15 | Train Loss: 0.1475 | Train Acc: 94.12% | Val Loss: 7.8861 | Val Acc: 57.35%
Epoch 8/15 | Train Loss: 0.1304 | Train Acc: 94.12% | Val Loss: 3.9643 | Val Acc: 58.82%
Epoch 9/15 | Train Loss: 0.1084 | Train Acc: 97.79% | Val Loss: 0.4820 | Val Acc: 82.35%
Epoch 10/15 | Train Loss: 0.0733 | Train Acc: 97.06% | Val Loss: 0.0682 | Val Acc: 97.06%
Epoch 11/15 | Train Loss: 0.0230 | Train Acc: 99.26% | Val Loss: 0.0031 | Val Acc

In [ ]:
# Ensure model is in evaluation mode before deploying
model.eval()

def predict(image):
    """
    Takes a PIL image, transforms it, and returns a dictionary of class confidences.
    """
    # The 'transform' object is the same one used for training
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image_tensor)
        # Apply softmax to get probabilities
        probabilities = torch.nn.functional.softmax(outputs, dim=1)[0]

        # Create a dictionary of class: confidence
        confidences = {train_data.classes[i]: float(probabilities[i]) for i in range(len(train_data.classes))}

    return confidences

In [ ]:
torch.save(model.state_dict(), "Resnet50_V3.pth")

In [ ]:
examples = [
    ['images/lehmus/lehmus6.jpg'],
    ['images/pihlaja/pihlaja23.jpg'],
    ['images/koivu/koivu15.jpg']
]
examples = [[os.path.join(images_dir, path[0])] for path in examples]


iface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload Hand Gesture"),
    outputs=gr.Label(num_top_classes=3, label="Predictions"),
    title="Rock, Paper, Scissors Classifier",
    description="Upload an image of a hand gesture (rock, paper, or scissors) to see the model's prediction. Trained on ResNet18.",
    examples=examples
)

# Launch the Gradio app
iface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f65a9a67ffa2b46897.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 745, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 353, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2112, in process_api
    inputs = await self.preprocess_data(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1754, in preprocess_data
    inputs_cached = await processing_utils.async_move_files_to_cache(
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/processing_utils.py", line 594, in async_move_files_to_cache
    return await client_utils.async_tr

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f65a9a67ffa2b46897.gradio.live
